### CS/ECE/ISyE 524 — Introduction to Optimization — Summer 2026 ###

# CitiBike Bike Rebalancing Optimization — Progress Report #

#### Junkai Zhang (jzhang3257@wisc.edu)
#### Zhenyu Gui (zgui6@wisc.edu)
#### Xiaonan Meng (xmeng67@wisc.edu)

### Table of Contents

1. [Introduction](#1-introduction)
2. [Mathematical Models](#2-mathematical-models)
3. [Work Completed](#3-work-completed)
4. [Work Remaining](#4-work-remaining)
5. [Issues & Concerns](#5-issues--concerns)

## 1. Introduction ##

CitiBike serves New York City, Jersey City, and Hoboken with over 200 stations. The core operational challenge is **bike rebalancing**: riders create imbalances — some stations overflow (surplus) while others run empty (deficit). Each night, an operator must deploy a fleet of trucks to reposition bikes from surplus to deficit stations, minimizing total travel distance while satisfying as much demand as possible.

This is a classic **minimum-cost network flow** problem with integer constraints — a natural application of the LP, MILP, and robust optimization techniques covered in ISyE 524. The problem belongs to the family of **Vehicle Routing Problems with Pickup and Delivery (VRPPD)** and was first formulated as a mathematical optimization problem by Raviv, Tzur, and Forma [1] in 2013.

**Data source**: CitiBike trip data for May 2026 (Jersey City), publicly available from [CitiBike System Data](https://citibikenyc.com/system-data). The raw dataset contains 94,995 trips across 224 stations. We aggregate net flow per station (arrivals − departures), classify stations as surplus (net > 0) or deficit (net < 0), and compute pairwise Haversine distances. The system has a structural supply shortage: 933 bikes available vs. 1,288 needed (355-bike gap).

**Key simplifying assumptions**:
- **Static single-period**: rebalancing occurs overnight; no time windows or traffic
- **Deterministic demand**: net flow computed from one month's data, treated as known
- **Haversine distances**: straight-line approximation (symmetric); road-network effects analyzed separately in Model 3
- **Single depot**: trucks start and end at a synthetic depot at the station centroid
- **Identical trucks**: all trucks have the same capacity Q = 50 bikes

We formulate four progressively more sophisticated models — from a simple LP transportation problem to a robust optimization under uncertainty — and validate each on 10, 25, and 50-station subsets before the full 224-station network.

[1] T. Raviv, M. Tzur, and I. A. Forma, "Static repositioning in a bike-sharing system: models and solution approaches," *EURO Journal on Transportation and Logistics*, 2(3), pp. 187–229, 2013.

## 2. Mathematical Models ##

We formulate four models of increasing sophistication. All models are implemented in Julia + JuMP with the HiGHS solver.

### Model 1 — Uncapacitated Transportation Problem (LP)

The simplest formulation: ignore trucks and treat surplus stations as pure sources and deficit stations as pure sinks. Decision variables $y_{ij} \ge 0$ represent bikes moved from surplus $i$ to deficit $j$.

$$\begin{aligned}
\underset{y}{\text{minimize}} \quad & \sum_{i \in S} \sum_{j \in D} d_{ij} \, y_{ij} \\
\text{subject to} \quad & \sum_{j \in D} y_{ij} \leq s_i \quad \forall i \in S \\
& \sum_{i \in S} y_{ij} = b_j \quad \forall j \in D \\
& y_{ij} \geq 0
\end{aligned}$$

**Total unimodularity**: The constraint matrix has at most one +1 and one −1 per column, guaranteeing integer optimal solutions without branching.

### Model 2 — Capacitated Vehicle Routing with MTZ (MILP)

Adds trucks with capacity $Q = 50$, depot start/end, and explicit route sequencing via MTZ subtour elimination. Key innovation: **arc sparsification** — each station connects only to its $M$ nearest neighbors plus the depot, reducing binary variables from $O(N^2 \cdot K)$ to $O(N \cdot M \cdot K)$ (a 92% reduction for the full dataset).

### Model 3 — Asymmetric Distance Sensitivity Analysis (LP)

Perturbs the symmetric Haversine matrix with random noise to simulate one-way streets and road geometry. Measures plan divergence, cost gap, and volume change between symmetric and asymmetric optimal plans.

### Model 4 — Robust Optimization under Uncertainty (LP)

Applies Bertsimas–Sim budgeted uncertainty [2] to protect against up to $\Gamma$ stations simultaneously realizing worst-case supply/demand deviations. The full dual derivation is provided in the final report.

[2] D. Bertsimas and M. Sim, "The price of robustness," *Operations Research*, 52(1), pp. 35–53, 2004.

## 3. Work Completed ##

### Data Processing
- Downloaded and cleaned CitiBike May 2026 trip data (94,995 trips, 224 stations)
- Aggregated net flow per station, computed Haversine distance matrix
- Generated 10, 25, and 50-station subsets for progressive validation

### Model Implementation

| Model | Status | Solver | Solve Time | Key Result |
|-------|--------|--------|------------|------------|
| Model 1 (LP) | ✅ Complete | HiGHS | <0.01s | 933 bikes, 200 arcs, 1.80 km avg |
| Model 2 (MILP) | ✅ Complete | HiGHS | 300s (10–50 stns) | Sparse: 8.6–32.0% gap; 287–414 bikes moved |
| Model 3 (Asym) | ✅ Complete | HiGHS | <0.01s | Haversine error ≤2% cost impact |
| Model 4 (Robust) | ✅ Complete | HiGHS | <0.01s | Infeasible for Γ≥1 (structural gap) |

### Model 2 Key Improvements
The original dense MILP formulation (304K binary variables) was computationally intractable. We implemented five improvements:
1. **Arc sparsification**: M-nearest neighbors per station, 92% variable reduction
2. **MTZ only on existing arcs**: avoids redundant constraints for unused trucks
3. **Tight big-M**: $\min(s_i, Q)$ instead of $s_i$ in pickup/dropoff linking
4. **Return empty**: trucks must return to depot empty
5. **Adaptive visit-once**: stations with supply/demand > Q allow multiple visits

### Results Summary (Model 2, Sparse Arc Formulation)

| Scale | M | Bin Vars | Trucks | Bikes | Unmet | Truck-km | Gap |
|-------|---|----------|--------|-------|-------|----------|-----|
| 10 stns | 5 | 140 | 2/2 | 287 | 289 | 31 | 8.6% |
| 25 stns | 8 | 750 | 3/3 | 414 | 452 | 49 | 16.1% |
| 50 stns | 10 | 2,400 | 4/4 | 338 | 734 | 59 | 32.0% |

### Final Report
- Draft complete with all four models, results, discussion, and conclusion
- Code and report are version-controlled on GitHub

## 4. Work Remaining ##

| Task | Estimated Time | Priority |
|------|---------------|----------|
| Model 2 full-scale (224 stns) run | 2–3 hours | High |
| Sensitivity analysis: truck capacity Q, fleet size K̄ | 2 hours | Medium |
| Improve visualizations (route maps, Pareto frontier) | 2 hours | Medium |
| Final report polishing & proofreading | 2 hours | High |
| Out-of-sample validation (July data) | 3 hours | Low |
| Export notebook to PDF for submission | 0.5 hours | High |

**Estimated remaining effort**: 10–12 hours over the next week. The computational bottleneck is Model 2 at full scale — the 300-second time limit is tight for 23K binary variables with MTZ constraints. We may need to increase the time limit or accept a larger optimality gap.

**Timeline**:
- Week 4- (current): Complete Model 2 full-scale run, finalize all results
- Future work: Polish report, add visualizations, export PDF

## 5. Issues & Concerns ##

### Issue 1: Model 2 Scaling (MTZ Weak LP Relaxation)

**Problem**: The MTZ subtour elimination formulation has a weak LP relaxation, causing the branch-and-bound tree to grow quickly. Even with arc sparsification (23K binaries for full dataset), the optimality gap widens with scale: 8.6% → 16.1% → 32.0% for 10→25→50 stations.

**Mitigation**: Arc sparsification reduces the problem size enough to find feasible solutions. We accept suboptimal solutions (within 5–32% gap) as a practical compromise. The results are still meaningful — trucks deliver 287–414 bikes with sensible multi-stop routes.

**Can we solve this?** Partially. Further tightening (e.g., lifted MTZ inequalities) could improve the LP relaxation, but a full solution would require a branch-and-cut approach (DFJ), which is beyond the scope of this course project.

### Issue 2: Structural Supply Shortage

**Problem**: The CitiBike system has a structural supply-demand gap (933 supply vs. 1,288 demand). Model 4 (Robust Optimization) becomes infeasible for any Γ > 0 because the system cannot absorb any uncertainty without external bike injection.

**Mitigation**: We treat this as a finding rather than a bug — the infeasibility is a genuine structural insight. The Bertsimas–Sim budget quantifies how much additional supply would be needed for robustness.

**Can we solve this?** Yes — by adding a "warehouse" node with unlimited (penalized) supply. This is a straightforward model extension.

### Issue 3: Haversine vs. Real Road Distances

**Problem**: All models use Haversine (straight-line) distances, which may differ from actual driving distances.

**Mitigation**: Model 3 demonstrates that perturbing distances by up to 20% changes total cost by ≤2%. For operational planning, Haversine is an adequate approximation. We acknowledge this as a limitation and suggest OSRM API integration as future work.

**Can we solve this?** Yes (OSRM API call), but Model 3 already shows the cost impact is minimal. Not a priority for this course project.

---

**Overall assessment**: The project is on track. All four models are implemented and tested. The main remaining work is running Model 2 at full scale and polishing the final report. We do not anticipate any blocking issues.